# World Bank Data Mart Project
## Notebook 01 - Raw Data Ingestion (Bronze Layer)

In this notebook we will:

1. Download World Bank indicator data using the World Bank API (via Python).
2. Combine multiple indicators into one “long format” dataset.
3. Save the result to disk as raw data (CSV + Parquet).
4. Perform the first cleaning step: remove non-country aggregate entities.

**Important idea:**  
We keep the raw data as close to the source as possible. This is our **Bronze layer**.
Later notebooks will create cleaned and modeled versions (Silver/Gold layers).

## Step 0 - Imports

We import:

- `requests` to call the World Bank API (download data from the web)
- `pandas` to store and manipulate the data as a table (DataFrame)

In [13]:
# Import the requests library for making HTTP requests (API calls)
import requests

# Import pandas for tabular data manipulation (DataFrames)
import pandas as pd

## Step 1 - Define a Downloader Function

World Bank data is accessed through an API endpoint.

Because the API returns data in **pages** (pagination), we write a function that:
- Requests page 1, page 2, page 3, etc.
- Extracts the useful fields from each record
- Combines all pages into one pandas DataFrame

This function downloads **one indicator at a time** (e.g., GDP only, inflation only).

In [14]:
def fetch_world_bank_indicator(indicator: str, start_year: int = 2000, end_year: int = 2024) -> pd.DataFrame:
    """
    Download ONE World Bank indicator (for ALL countries/entities) for a given year range.

    Parameters
    ----------
    indicator : str
        The official World Bank indicator code (example: "NY.GDP.MKTP.CD").
    start_year : int
        First year to download (inclusive).
    end_year : int
        Last year to download (inclusive).

    Returns
    -------
    pd.DataFrame
        A DataFrame where each row is ONE measurement:
        - country_iso3 : 3-letter ISO code (e.g., "USA")
        - country      : country/entity name
        - year         : year (int)
        - indicator_code : the indicator code (same as input)
        - value        : numeric value from World Bank
    """

    # We'll store each record (row) we download in this list.
    # Later we convert this list of dicts into a DataFrame.
    records = []

    # The World Bank API returns results in pages.
    # We'll start at page 1 and loop until we reach the last page.
    page = 1

    # Loop indefinitely; we will "break" once we've downloaded all pages.
    while True:
        # Build the World Bank API URL for the current page.
        #
        # Explanation of parameters:
        # - country/all: request data for all entities
        # - indicator/{indicator}: which indicator we want (GDP, inflation, etc.)
        # - format=json: response comes back as JSON
        # - per_page=20000: request up to 20,000 rows in each page
        # - page={page}: which page number we are currently downloading
        # - date=start:end: year range filter
        url = (
            f"https://api.worldbank.org/v2/country/all/indicator/{indicator}"
            f"?format=json&per_page=20000&page={page}&date={start_year}:{end_year}"
        )

        # Make the web request (HTTP GET).
        # timeout=60 means: if it takes >60 seconds, stop and raise an error.
        r = requests.get(url, timeout=60)

        # If the server responded with an error code (like 404, 500), raise an exception.
        r.raise_for_status()

        # Convert the response body from JSON text into Python objects (lists/dicts).
        data = r.json()

        # World Bank JSON response structure:
        # - data[0] = metadata (includes total pages)
        # - data[1] = list of records (the actual data rows)
        #
        # If data[1] is missing or None, there's nothing else to download.
        if len(data) < 2 or data[1] is None:
            break

        # Iterate through each record in this page and extract the fields we want.
        for row in data[1]:
            # Build a clean row dictionary with only the columns we care about.
            records.append({
                # ISO3 code (3-letter country code); may be None for aggregates
                "country_iso3": row.get("countryiso3code"),

                # Country/entity name is nested inside row["country"]["value"]
                # If row.get("country") is None, (row.get("country") or {}) becomes {},
                # and .get("value") returns None safely (no crash).
                "country": (row.get("country") or {}).get("value"),

                # Year comes back as a string like "2024" in the "date" field.
                # Convert to int for easier filtering/sorting later.
                "year": int(row["date"]) if row.get("date") else None,

                # Keep the indicator code we requested so we know what metric this row represents.
                "indicator_code": indicator,

                # The numeric value of the indicator for that country/year.
                "value": row.get("value"),
            })

        # Pagination handling:
        # data[0]["pages"] tells us how many total pages exist for this query.
        pages = int(data[0]["pages"])

        # If we've reached the last page, stop.
        if page >= pages:
            break

        # Otherwise, move on to the next page.
        page += 1

    # Convert list of row dictionaries into a pandas DataFrame and return it.
    return pd.DataFrame(records)

## Step 2 - Choose Indicators to Download

The World Bank identifies indicators by codes.

We define a small set of indicators that are useful for economic analysis and dashboards:

- GDP (total)
- GDP per capita
- Inflation %
- Unemployment %
- Population

We also map each official code to a friendly name we will use later.

In [15]:
# Dictionary mapping:
# World Bank indicator code -> friendly name we want to use later
indicators = {
    "NY.GDP.MKTP.CD": "gdp_usd",              # GDP (current US$)
    "NY.GDP.PCAP.CD": "gdp_per_capita_usd",   # GDP per capita (current US$)
    "FP.CPI.TOTL.ZG": "inflation_pct",        # Inflation, consumer prices (annual %)
    "SL.UEM.TOTL.ZS": "unemployment_pct",     # Unemployment (% of total labor force)
    "SP.POP.TOTL": "population",              # Population, total
}

## Step 3 - Download All Indicators and Combine Into One Table

We now:

1. Loop through our indicator list
2. Download each indicator using our function
3. Add a column (`indicator_name`) so we can label each metric clearly
4. Concatenate (stack) them into one large DataFrame called `raw`

This creates a **long-format dataset** where each row is:

**(country, year, indicator, value)**

In [16]:
# We'll store each indicator's DataFrame in this list.
dfs = []

# Loop through each indicator code + friendly name.
for code, name in indicators.items():
    # Print progress so we know which indicator is currently downloading.
    print("Downloading:", code, name)

    # Download the indicator data from the World Bank API for years 2000–2024.
    df = fetch_world_bank_indicator(code, 2000, 2024)

    # Add a friendly indicator name column (e.g., "gdp_usd") to each row.
    # This makes pivoting/reshaping easier later.
    df["indicator_name"] = name

    # Append the DataFrame for this indicator to our list.
    dfs.append(df)

# Combine all indicator DataFrames into one table by stacking rows.
# ignore_index=True resets row numbers from 0..N-1
raw = pd.concat(dfs, ignore_index=True)

# Preview the first few rows
raw.head()

Downloading: NY.GDP.MKTP.CD gdp_usd
Downloading: NY.GDP.PCAP.CD gdp_per_capita_usd
Downloading: FP.CPI.TOTL.ZG inflation_pct
Downloading: SL.UEM.TOTL.ZS unemployment_pct
Downloading: SP.POP.TOTL population


,country_iso3,country,year,indicator_code,value,indicator_name
0,AFE,Africa Eastern and Southern,2024,NY.GDP.MKTP.CD,1.242694e+12,gdp_usd
1,AFE,Africa Eastern and Southern,2023,NY.GDP.MKTP.CD,1.179359e+12,gdp_usd
2,AFE,Africa Eastern and Southern,2022,NY.GDP.MKTP.CD,1.228968e+12,gdp_usd
3,AFE,Africa Eastern and Southern,2021,NY.GDP.MKTP.CD,1.114145e+12,gdp_usd
4,AFE,Africa Eastern and Southern,2020,NY.GDP.MKTP.CD,9.386076e+11,gdp_usd


## Step 4 - Sanity Check the Download

We check:
- the shape of the dataset (rows, columns)

This tells us if we downloaded a reasonable amount of data.

In [18]:
# Show the number of rows and columns in the raw dataset.
raw.shape

(33250, 6)

## Step 5 - Save the Raw Data (Bronze Layer)

We save the raw dataset to `data_raw/` in two formats:

- CSV: easy to view, universal
- Parquet: compressed + fast, used in real data pipelines

We do NOT commit these files to GitHub (they are ignored by `.gitignore`).

In [20]:
# pathlib makes file paths easier and cross-platform friendly
from pathlib import Path

# Ensure the output folder exists
Path("../data_raw").mkdir(exist_ok=True)

# Save to CSV (human-readable)
raw.to_csv("../data_raw/world_bank_indicators_raw.csv", index=False)

# Save to Parquet (faster, compressed)
raw.to_parquet("../data_raw/world_bank_indicators_raw.parquet", index=False)

print("Saved raw data!")

Saved raw data!


## Step 6 - Inspect Entities (Countries vs Aggregates)

The World Bank API returns *more than just countries*.

It can include:
- Regions (e.g., "Africa Eastern and Southern")
- Income groups (e.g., "High income")
- World totals

Before we build a dashboard or SQL data mart, we want to remove non-country aggregates.

First, we inspect:
- number of unique entity names
- a sample of entity names

In [21]:
# Count how many unique entity names appear in the raw dataset.
# This includes countries AND aggregate regions/groups.
raw["country"].nunique()

266

In [22]:
# Show a small sample of entity names (alphabetical) to spot aggregates.
raw["country"].sort_values().unique()[:20]

<ArrowStringArray>
[                'Afghanistan', 'Africa Eastern and Southern',
  'Africa Western and Central',                     'Albania',
                     'Algeria',              'American Samoa',
                     'Andorra',                      'Angola',
         'Antigua and Barbuda',                  'Arab World',
                   'Argentina',                     'Armenia',
                       'Aruba',                   'Australia',
                     'Austria',                  'Azerbaijan',
                'Bahamas, The',                     'Bahrain',
                  'Bangladesh',                    'Barbados']
Length: 20, dtype: str

## Step 7 - Remove Aggregate Entities (First Cleaning Step)

We filter to keep only rows where:
- `country_iso3` exists (not null)
- `country_iso3` is exactly 3 characters long

Reason:
- Real countries have ISO3 codes like "USA", "FRA"
- Many aggregates have missing or irregular codes

This creates `clean`, which is our first cleaned dataset.

In [23]:
# Filter out aggregates by keeping only rows with a valid ISO3 code.
# Conditions:
# 1) ISO3 code is not null
# 2) ISO3 code has exactly 3 characters
clean = raw[
    raw["country_iso3"].notna() &             # Must not be missing
    (raw["country_iso3"].str.len() == 3)      # Must be a 3-letter code
].copy()

# Show the new shape after filtering
clean.shape

(32625, 6)

## Step 8 - Compare Unique Entity Counts Before vs After Cleaning

We compare how many unique entity names exist before and after filtering.
This confirms that aggregates were removed.

In [24]:
# Compare number of unique entity names before and after cleaning.
print("Before:", raw["country"].nunique())
print("After:", clean["country"].nunique())

Before: 266
After: 261


## Optional Check - Which Entities Were Removed?

We removed non-country aggregates by filtering on valid ISO3 codes.

This optional step computes the **set difference** between:
- entities in the raw dataset
- entities in the cleaned dataset

So we can explicitly see which names were removed.

In [25]:
# Get the set of unique entity names in the raw dataset
raw_entities = set(raw["country"].dropna().unique())

# Get the set of unique entity names in the cleaned dataset
clean_entities = set(clean["country"].dropna().unique())

# Entities that exist in raw but NOT in clean are what we removed
removed_entities = sorted(raw_entities - clean_entities)

# Print count and list of removed entities
print("Number of removed entities:", len(removed_entities))
print("Removed entities:")
for name in removed_entities:
    print("-", name)

Number of removed entities: 5
Removed entities:
- High income
- Low income
- Lower middle income
- Not classified
- Upper middle income


Looks good. Those 5 are income-group aggregates, not countries. If we kept them, they’d pollute comparisons and dashboards.

## Step 9 - Remove Rows With Missing Indicator Values

The World Bank dataset contains missing values because some countries do not report
every indicator every year.

At this stage, our dataset is still in **long format**:
- each row = one (country, year, indicator) measurement

Before reshaping the data (pivoting long → wide), we remove rows where the
indicator `value` is missing.

**Important note:**
After we pivot to wide format, some country-year rows may still be missing certain metrics.
That is normal. This step only removes rows where the value is completely missing in the long table.

In [26]:
# Step 9A: Count missing values in the indicator value column

# Count how many rows have value == NaN (missing)
missing_value_rows = clean["value"].isna().sum()

# Print the missing count and total size BEFORE dropping
print("Rows with missing 'value' BEFORE dropping:", missing_value_rows)
print("Total rows BEFORE dropping:", len(clean))


# Step 9B: Drop rows where the indicator value is missing

# We only drop rows where "value" is missing.
# We keep the rest of the row structure (country, year, indicator info).
clean_nonnull = clean.dropna(subset=["value"]).copy()

# Print the size AFTER dropping
print("Total rows AFTER dropping:", len(clean_nonnull))

# Calculate and show how many rows were removed
print("Rows removed:", len(clean) - len(clean_nonnull))

Rows with missing 'value' BEFORE dropping: 2052
Total rows BEFORE dropping: 32625
Total rows AFTER dropping: 30573
Rows removed: 2052


### Step 9C - Quick Sanity Checks

After dropping null indicator values, we quickly confirm:

- `value` has no missing values
- We still have the same main columns we expect

In [27]:
# Confirm there are now 0 missing values in the 'value' column
print("Missing 'value' AFTER dropping:", clean_nonnull["value"].isna().sum())

# Show the column names so we confirm the dataset structure is intact
print("Columns in clean_nonnull:", list(clean_nonnull.columns))

# Preview a few rows to visually confirm data looks correct
clean_nonnull.head()

Missing 'value' AFTER dropping: 0
Columns in clean_nonnull: ['country_iso3', 'country', 'year', 'indicator_code', 'value', 'indicator_name']


,country_iso3,country,year,indicator_code,value,indicator_name
0,AFE,Africa Eastern and Southern,2024,NY.GDP.MKTP.CD,1.242694e+12,gdp_usd
1,AFE,Africa Eastern and Southern,2023,NY.GDP.MKTP.CD,1.179359e+12,gdp_usd
2,AFE,Africa Eastern and Southern,2022,NY.GDP.MKTP.CD,1.228968e+12,gdp_usd
3,AFE,Africa Eastern and Southern,2021,NY.GDP.MKTP.CD,1.114145e+12,gdp_usd
4,AFE,Africa Eastern and Southern,2020,NY.GDP.MKTP.CD,9.386076e+11,gdp_usd


## Step 10 - Confirm Data Types

Before reshaping the dataset (pivoting long → wide), we confirm that:

- `year` is numeric (integer)
- `value` is numeric (float or integer)

If these are accidentally stored as strings, pivoting and calculations
can behave incorrectly or fail.

In [28]:
# Show data types for each column in the cleaned dataset
# We care most about:
# - year: should be int (or convertible to int)
# - value: should be float or int
clean_nonnull.dtypes

country_iso3          str
country               str
year                int64
indicator_code        str
value             float64
indicator_name        str
dtype: object

## Step 11 - Reshape Data (Long to Wide)

Currently, our dataset is in long format:
- each row = one metric for one country in one year

We now pivot the table so that:
- each row = one country-year
- each metric becomes its own column

This structure is better for:
- SQL data marts
- BI dashboards
- analytical modeling

In [29]:
# Step 11A: Pivot the dataset from long format to wide format

# Use pandas pivot_table:
# index = columns that define a unique row (country + year)
# columns = indicator_name (these will become new columns)
# values = numeric values to fill into those new columns
# aggfunc = "first" because each country-year-indicator should be unique
wide = clean_nonnull.pivot_table(
    index=["country_iso3", "country", "year"],
    columns="indicator_name",
    values="value",
    aggfunc="first"
).reset_index()

# After pivoting, indicator names become column headers.
# Resetting index turns country/year back into normal columns.

# Remove the column name grouping label that pandas adds
wide.columns.name = None

# Show first few rows
wide.head()

,country_iso3,country,year,gdp_per_capita_usd,gdp_usd,inflation_pct,population,unemployment_pct
0,ABW,Aruba,2000,20681.023027,1.873453e+09,4.044021,90588.0,NaN
1,ABW,Aruba,2001,20740.132583,1.896457e+09,2.883604,91439.0,NaN
2,ABW,Aruba,2002,21307.248251,1.961844e+09,3.315247,92074.0,NaN
3,ABW,Aruba,2003,21949.485996,2.044112e+09,3.656365,93128.0,NaN
4,ABW,Aruba,2004,23700.631990,2.254831e+09,2.529129,95138.0,NaN


### Step 11B - Inspect the Wide Dataset

We confirm:
- shape of the new dataset
- column names
- whether pivot worked correctly

In [30]:
# Show shape (rows, columns)
print("Wide dataset shape:", wide.shape)

# Show column names
print("Columns:", list(wide.columns))

# Preview data
wide.head()

Wide dataset shape: (6525, 8)
Columns: ['country_iso3', 'country', 'year', 'gdp_per_capita_usd', 'gdp_usd', 'inflation_pct', 'population', 'unemployment_pct']


,country_iso3,country,year,gdp_per_capita_usd,gdp_usd,inflation_pct,population,unemployment_pct
0,ABW,Aruba,2000,20681.023027,1.873453e+09,4.044021,90588.0,NaN
1,ABW,Aruba,2001,20740.132583,1.896457e+09,2.883604,91439.0,NaN
2,ABW,Aruba,2002,21307.248251,1.961844e+09,3.315247,92074.0,NaN
3,ABW,Aruba,2003,21949.485996,2.044112e+09,3.656365,93128.0,NaN
4,ABW,Aruba,2004,23700.631990,2.254831e+09,2.529129,95138.0,NaN


## Step 12 - Save the Cleaned & Reshaped Dataset (Silver Layer)

We now save the wide-format dataset as our cleaned dataset.

This dataset will be used for:

- SQL data mart creation
- Power BI dashboards
- Analytical queries

We store it separately from raw data.

In [32]:
from pathlib import Path

# Ensure processed data folder exists
Path("../data_processed").mkdir(exist_ok=True)

# Save to CSV
wide.to_csv("../data_processed/world_bank_indicators_wide.csv", index=False)

# Save to Parquet
wide.to_parquet("../data_processed/world_bank_indicators_wide.parquet", index=False)

print("Saved processed data (Silver layer).")

Saved processed data (Silver layer).


## Notebook 01 Summary

In this notebook we:

- Downloaded economic indicators from the World Bank API
- Stored raw data (Bronze layer)
- Removed aggregate entities
- Dropped null indicator rows
- Enforced numeric types
- Pivoted long → wide format
- Saved cleaned dataset (Silver layer)

The resulting dataset is ready for SQL modeling and BI visualization.